# Desafio Técnico Artifact AI Engineer 2026

In [1]:
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, Markdown

In [2]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

# Configuração do LLM apontando para o LM Studio
# llm = ChatOpenAI(
#     base_url="http://localhost:1234/v1",
#     api_key="lm-studio",
#     model="qwen/qwen3.5-9b",
#     temperature=0
# )

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

## Agente SQL

In [3]:
import pandas as pd
from sqlalchemy import create_engine

# 1. Criação do Banco de Dados SQLite local
engine = create_engine("sqlite:///loja_musica.db")

# Dicionário mapeando o nome do arquivo para o nome da tabela no SQL
arquivos_csv = {
    "categories": "data/desafio_tecnico_ai_eng - categories.csv",
    "customers": "data/desafio_tecnico_ai_eng - customers.csv",
    "order_items": "data/desafio_tecnico_ai_eng - order_items.csv",
    "orders": "data/desafio_tecnico_ai_eng - orders.csv",
    "products": "data/desafio_tecnico_ai_eng - products.csv",
    "promotions": "data/desafio_tecnico_ai_eng - promotions.csv"
}

# Leitura dos CSVs e inserção no banco de dados
# for nome_tabela, caminho_arquivo in arquivos_csv.items():
#     df = pd.read_csv(caminho_arquivo)
#     # Salva o DataFrame como uma tabela SQL. replace garante que reescreva se rodar de novo.
#     df.to_sql(nome_tabela, engine, index=False, if_exists="replace")

# print("Banco de dados loja_musica.db criado com sucesso!")


In [9]:

from langchain_community.utilities import SQLDatabase
from langchain_core.prompts import PromptTemplate
from langchain_community.agent_toolkits import create_sql_agent

# Conexão do LangChain ao banco de dados
db = SQLDatabase(engine=engine)

sql_system_rules_prompt = PromptTemplate.from_template("""Você é um assistente de IA especializado em consultas SQL para uma loja de instrumentos musicais.

    Responda sempre em Português Brasileiro.
    
    FILTRAGEM EXATA: Se o usuário pedir um instrumento específico (ex: "violão"), você DEVE usar a cláusula WHERE para filtrar nas colunas name e description do produto. (usando LIKE '%nome%').
    
    Para consultas de produtos, leve em consideração o preço, a categoria na qual o produto está cadastrado, a descrição do produto e a disponibilidade em estoque.
    Se necessário faça junções entre as tabelas para obter informações completas sobre produtos e suas respectivas categorias, pedidos, clientes e promoções.
    Responda a pergunta com detalhes e informações do banco de dados:

    {question}

{question}
""")

# Criação do Agente SQL
sql_agent = create_sql_agent(
    llm,
    db=db,
    agent_type="openai-tools",
    # verbose=True # Mantém o log ativo para ver a query SQL gerada no terminal,
)

sql_chain = sql_system_rules_prompt | sql_agent

In [ ]:
resposta = sql_chain.invoke({"question": "Qual foi o cliente que fez o pedido mais caro?"})
display(Markdown(resposta["output"]))

O cliente que fez o pedido mais caro foi **Pedro Henrique Oliveira**, com um total de **R$ 11.499,00**.

In [10]:
resposta = sql_chain.invoke({"question": "Quais são as três categorias com mais produtos cadastrados?"})
display(Markdown(resposta["output"]))

As três categorias com mais produtos cadastrados são:

1. **Violões** - 35 produtos
2. **Ukuleles** - 15 produtos
3. **Guitarras** - 5 produtos

In [ ]:
resposta = sql_chain.invoke({"question": "Qual são os três violões mais baratos?"} )
display(Markdown(resposta["output"]))

Os três violões mais baratos disponíveis na loja são:

1. **Tagima Memphis AC-39 Nylon Natural** - R$ 429,90
2. **Yamaha C40 Nylon Natural** - R$ 599,90
3. **Yamaha C70 Nylon Natural** - R$ 849,00

Esses violões são ideais para iniciantes e estudantes, com características que atendem a diferentes níveis de habilidade.

In [9]:
resposta = sql_chain.invoke({"question": "Quais as três categorias de produtos mais vendidas?"})
display(Markdown(resposta["output"]))

As três categorias de produtos mais vendidas na loja de instrumentos musicais são:

1. **Violões** - 13 unidades vendidas
2. **Ukuleles** - 9 unidades vendidas
3. **Baixos** - 1 unidade vendida

Essas informações refletem as categorias com maior volume de vendas, indicando a preferência dos clientes.

## PDF RAG

In [39]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Configura o modelo de embedding MULTILÍNGUE
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

diretorio_chroma = "./chroma_db_pt"

# 2. Lógica para não reprocessar o PDF à toa
if not os.path.exists(diretorio_chroma):
    print("Processando PDF e criando banco vetorial pela primeira vez...")
    loader = PyPDFLoader("data/políticas_da_loja.pdf")
    documentos = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    chunks = text_splitter.split_documents(documentos)
    
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=diretorio_chroma
    )
else:
    print("Carregando banco vetorial existente do disco...")
    vector_store = Chroma(
        persist_directory=diretorio_chroma, 
        embedding_function=embeddings
    )

retriever = vector_store.as_retriever(search_kwargs={"k": 4})


# 4. Cadeia RAG
template = """Use os trechos de contexto abaixo para responder à pergunta, se possível detalhadamente. 
Se a resposta não estiver no contexto, diga que não sabe. Não invente informações.

Contexto:
{context}

Pergunta: {question}
"""
prompt = PromptTemplate.from_template(template)

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4117.93it/s]


Carregando banco vetorial existente do disco...


## Agente de roteamento

In [13]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# 1. Prompt de Classificação
prompt_classificacao = PromptTemplate.from_template(
    """Você é um roteador inteligente para uma loja de instrumentos musicais.
    Analise a pergunta e decida qual sistema deve responder.
    
    Sistemas:
    - "SQL": Para perguntas sobre vendas, clientes, pedidos, catálogo de produtos e promoções.
    - "PDF": Para perguntas sobre regras da loja, políticas de devolução, trocas e garantias.
    
    Pergunta: {question}
    
    Responda APENAS com a palavra SQL ou PDF:"""
)

cadeia_classificacao = prompt_classificacao | llm | StrOutputParser()

def delegar_tarefa(info):
    classificacao = info["classificacao"].strip().upper()
    pergunta = info["question"]
    
    print(f"\n[Agente Mestre delegou para: {classificacao}]")
    
    if "SQL" in classificacao:
        # Chama o agente do banco de dados
        resposta_sql = sql_chain.invoke(pergunta)
        return resposta_sql["output"]
    else:
        # Chama a chain do ChromaDB
        return rag_chain.invoke(pergunta)

# 3. Construção do Agente Mestre
agente_mestre = (
    {"classificacao": cadeia_classificacao, "question": RunnablePassthrough()}
    | RunnableLambda(delegar_tarefa)
)

## Agente Chatbot

In [31]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

chatbot_prompt = PromptTemplate.from_template(
    """Você é um chatbot atendente simpático e prestativo de uma loja de instrumentos musicais.
    Responda à pergunta de forma educada e amigável com os dados que foram obtidos pelo sistema.
    
    Pergunta: {question}

    Dados obtidos: {obtained_data}
    """
)

chatbot_chain = ( 
    { "question": RunnablePassthrough(), "obtained_data": agente_mestre } 
    | chatbot_prompt 
    | llm 
    | StrOutputParser() )


In [32]:
user_question = "Quanto custa o violão mais barato?"
resposta_final = chatbot_chain.invoke(user_question)
display(Markdown(f"""**Pergunta:** {user_question}

**Resposta:**
{resposta_final}"""))


[Agente Mestre delegou para: SQL]


**Pergunta:** Quanto custa o violão mais barato?

**Resposta:**
Olá! 😊 Fico feliz em ajudar! O violão mais barato que temos em nossa loja custa R$ 429,90. Se precisar de mais informações ou ajuda para escolher um modelo, estou à disposição! 🎶

In [22]:
user_question = "Me arrependi da minha compra, posso devolver meu pedido?"
resposta_final = chatbot_chain.invoke(user_question)
display(Markdown(f"""**Pergunta:** {user_question}

**Resposta:**
{resposta_final}"""))


[Agente Mestre delegou para: PDF]


**Pergunta:** Me arrependi da minha compra, posso devolver meu pedido?

**Resposta:**
Olá! 😊 Que bom que você entrou em contato! Entendo que às vezes as coisas não saem como planejado. Mas não se preocupe, você pode devolver seu pedido em caso de arrependimento, de acordo com as nossas políticas. 

Se precisar de ajuda com o processo de devolução ou tiver mais alguma dúvida, estou aqui para te ajudar! 🎶✨

In [23]:
user_question = "Como faço para devolver um produto que comprei?"
resposta_final = chatbot_chain.invoke(user_question)
display(Markdown(f"""**Pergunta:** {user_question}

**Resposta:**
{resposta_final}"""))


[Agente Mestre delegou para: PDF]


**Pergunta:** Como faço para devolver um produto que comprei?

**Resposta:**
Olá! 😊 Que bom que você entrou em contato! Para devolver um produto que você comprou, é super fácil! Basta seguir esses passos:

1. **Solicite a devolução** em até 7 dias corridos após receber o produto. E o melhor: não precisa dar justificativa!
2. Certifique-se de que o produto esteja na **embalagem original**, sem sinais de uso, e que todos os acessórios e manuais estejam incluídos.
3. Assim que recebermos o produto, o **reembolso** será feito na mesma forma de pagamento original em até 10 dias úteis. 
4. E não se preocupe com o frete de devolução, ele será coberto pela nossa loja em caso de arrependimento! 🎉

Se precisar de mais alguma coisa ou tiver dúvidas, estou aqui para ajudar! 🎶✨

In [43]:
user_question = "Qual o endereço da loja?"
resposta_final = chatbot_chain.invoke(user_question)

display(Markdown(f"""**Pergunta:** {user_question}

**Resposta:**
{resposta_final}"""))


[Agente Mestre delegou para: PDF]


**Pergunta:** Qual o endereço da loja?

**Resposta:**
Olá! 😊 Fico feliz em ajudar! O endereço da nossa loja é Rua 14 de Maio, 3200 — Centro, Campo Grande – MS – CEP 79202-333. Se precisar de mais alguma informação ou tiver alguma dúvida, é só me chamar! 🎶

## Sessão com memória

In [41]:
# Dicionário para armazenar o estado (resumo) de cada sessão na memória RAM
resumos_sessao = {}

def chat_with_memory(pergunta_usuario: str, session_id: str) -> str:
    # 1. Recupera o resumo atual do usuário
    resumo_atual = resumos_sessao.get(session_id, "")
    
    # 2. Injeta o contexto no prompt do agente
    if resumo_atual:
        input_agente = f"Contexto histórico:\n{resumo_atual}\n\nNova pergunta: {pergunta_usuario}"
    else:
        input_agente = pergunta_usuario
        
    # 3. Invoca o agente existente
    resposta = chatbot_chain.invoke(input_agente)
    
    # 4. Atualiza o resumo em background usando o seu LLM local
    prompt_resumo = f"""
    Resumo anterior: {resumo_atual}
    Novo input do usuário: {pergunta_usuario}
    Nova resposta da IA: {resposta}
    
    Sintetize o contexto atualizado desta conversa de forma concisa para servir de memória futura.
    """
    
    # Chama o llm puramente para resumir
    novo_resumo = llm.invoke(prompt_resumo).content
    resumos_sessao[session_id] = novo_resumo
    
    return resposta

In [35]:
pergunta1 = "Qual o cliente que fez o pedido mais recente?"
pergunta2 = "Quais produtos ele comprou?"
session_id = "session_01"

resposta1 = chat_with_memory(pergunta1, session_id)
resposta2 = chat_with_memory(pergunta2, session_id)

display(Markdown(f"""**Pergunta:** {pergunta1}

**Resposta:**
{resposta1}"""))

display(Markdown(f"""**Pergunta:** {pergunta2}

**Resposta:**
{resposta2}"""))


[Agente Mestre delegou para: SQL]

[Agente Mestre delegou para: SQL]


**Pergunta:** Qual o cliente que fez o pedido mais recente?

**Resposta:**
Olá! 😊 O cliente que fez o pedido mais recente é a Camila Duarte Nascimento, e ela realizou o pedido no dia 22 de março de 2026. Se precisar de mais alguma informação ou ajuda, estou aqui para ajudar! 🎶

**Pergunta:** Quais produtos ele comprou?

**Resposta:**
Olá! 😊 Camila Duarte Nascimento comprou um **Tagima Memphis AC-39 Nylon Natural** em seu pedido mais recente, realizado em 22 de março de 2026. O preço do violão é de **R$ 429,90**. Esse violão clássico é ideal para quem está começando, com cordas de nylon e um corpo leve e confortável, perfeito para longas sessões de estudo.

Se precisar de mais informações ou tiver outras perguntas, estou aqui para ajudar! 🎶

In [36]:
user_question = "Quais opções de violões disponíveis vocês têm custando até R$ 1000?"
resposta_final = chat_with_memory(user_question, session_id)

display(Markdown(f"""**Pergunta:** {user_question}

**Resposta:**
{resposta_final}"""))


[Agente Mestre delegou para: SQL]


**Pergunta:** Quais opções de violões disponíveis vocês têm custando até R$ 1000?

**Resposta:**
Olá! 😊 Fico feliz em ajudar! Temos algumas ótimas opções de violões disponíveis até R$ 1000. Aqui estão elas:

1. **Tagima Memphis AC-39 Nylon Natural**
   - **Preço:** R$ 429,90
   - **Descrição:** Violão clássico com cordas de nylon, perfeito para quem está começando. Corpo leve e confortável para tocar.

2. **Yamaha C40 Nylon Natural**
   - **Preço:** R$ 599,90
   - **Descrição:** Violão clássico com tampo em Spruce e fundo/laterais em Meranti. Ideal para estudantes e iniciantes.

3. **Yamaha C70 Nylon Natural**
   - **Preço:** R$ 849,00
   - **Descrição:** Violão clássico de nível intermediário com tampo em Spruce sólido. Acabamento de alta qualidade.

Se precisar de mais informações sobre algum desses produtos ou se tiver alguma outra dúvida, estou aqui para ajudar! 🎶